In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/yzta-2026-datathon/sample_submission.csv
/kaggle/input/competitions/yzta-2026-datathon/test_x.csv
/kaggle/input/competitions/yzta-2026-datathon/train.csv


In [2]:
import pandas as pd
import numpy as np
from catboost import CatBoostRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
import warnings

warnings.filterwarnings('ignore')

# --- 1. VERİ OKUMA ---
train = pd.read_csv('/kaggle/input/competitions/yzta-2026-datathon/train.csv')
test = pd.read_csv('/kaggle/input/competitions/yzta-2026-datathon/test_x.csv')

# --- 2. GELİŞMİŞ PREPROCESSING & GÖRECELİ ÖZELLİKLER ---
def prepare_optimized_data(df, train_ref=None):
    df = df.copy()
    
    # A. Eksik Veri Doldurma (Train referanslı)
    cat_cols = ['meslek', 'kronotip', 'ruh_sagligi_durumu']
    for col in cat_cols:
        df[col] = df[col].fillna('Bilinmiyor')
        
    if train_ref is not None:
        stres_med = train_ref.groupby('meslek')['stres_skoru'].median()
        df['stres_skoru'] = df['stres_skoru'].fillna(df['meslek'].map(stres_med))
        df['vucut_kitle_indeksi'] = df['vucut_kitle_indeksi'].fillna(train_ref['vucut_kitle_indeksi'].median())
    else:
        df['stres_skoru'] = df['stres_skoru'].fillna(df.groupby('meslek')['stres_skoru'].transform('median'))
        df['vucut_kitle_indeksi'] = df['vucut_kitle_indeksi'].fillna(df['vucut_kitle_indeksi'].median())
    
    df['stres_skoru'] = df['stres_skoru'].fillna(df['stres_skoru'].median())
    df['uyku_oncesi_kafein_mg'] = df['uyku_oncesi_kafein_mg'].fillna(0)

    # B. DOMAIN ÖZELLİKLERİ
    df['stres_krizi_siniri'] = (df['stres_skoru'] > 5.5).astype(int)
    df['tam_saglikli_mi'] = (df['ruh_sagligi_durumu'] == 'Saglikli').astype(int)
    df['total_quality_sleep'] = df['rem_yuzdesi'] + df['derin_uyku_yuzdesi']
    df['sleep_disruption_ratio'] = df['gecelik_uyanma_sayisi'] / (df['total_quality_sleep'] + 1)
    df['is_stres_yukü'] = df['gunluk_calisma_saati'] * df['stres_skoru']

    # C. YENİ: GÖRECELİ (RELATIVE) ÖZELLİKLER (1.20 Altına İniş Anahtarı)
    # Kişinin kendi meslek grubundaki diğer insanlara göre stres durumu
    df['meslek_stres_farki'] = df['stres_skoru'] - df.groupby('meslek')['stres_skoru'].transform('mean')
    # Yaşa göre uyku kalitesi verimliliği
    df['yas_uyku_verimi'] = df['total_quality_sleep'] / (df['yas'] + 1)
    # Vücut Kitle İndeksi ve Aktivite Dengesi
    df['vki_aktivite_orani'] = df['gunluk_adim_sayisi'] / (df['vucut_kitle_indeksi'] + 1)

    return df

train = prepare_optimized_data(train)
test = prepare_optimized_data(test, train_ref=train)

# --- 3. EĞİTİM HAZIRLIĞI ---
cat_features = ['cinsiyet', 'meslek', 'ulke', 'kronotip', 'ruh_sagligi_durumu', 'mevsim', 'gun_tipi']
TARGET = 'bilissel_performans_skoru'

X = train.drop(['id', TARGET], axis=1)
y = train[TARGET]
X_test = test.drop(['id'], axis=1)

# CatBoost Parametreleri
cat_params = {
    'iterations': 5000,
    'learning_rate': 0.012, # Daha hassas öğrenme için biraz düşürüldü
    'depth': 7,
    'l2_leaf_reg': 5, # Overfitting'e karşı regülarizasyon artırıldı
    'eval_metric': 'RMSE',
    'random_seed': 42,
    'verbose': 0,
    'early_stopping_rounds': 300,
    'bagging_temperature': 0.2
}

# --- 4. SEED AVERAGING + 10-FOLD CV ---
seeds = [42, 2026, 1903]
final_test_preds = np.zeros(len(X_test))
oof_preds = np.zeros(len(X))

for seed in seeds:
    print(f"\n--- Eğitim Başlıyor (Seed: {seed}) ---")
    cat_params['random_seed'] = seed
    kf = KFold(n_splits=10, shuffle=True, random_state=seed)
    
    for fold, (train_idx, val_idx) in enumerate(kf.split(X, y)):
        X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]
        
        model = CatBoostRegressor(**cat_params)
        model.fit(X_tr, y_tr, eval_set=(X_val, y_val), cat_features=cat_features)
        
        oof_preds[val_idx] += model.predict(X_val) / len(seeds)
        final_test_preds += model.predict(X_test) / (len(seeds) * 10)
        
    print(f"Seed {seed} tamamlandı.")

cv_rmse = np.sqrt(mean_squared_error(y, oof_preds))
print(f"\n>>> YENİ OPTİMİZE CV RMSE: {cv_rmse:.5f} <<<")

# --- 5. KAYIT ---
# RMSE'yi iyileştirmek için uç değerleri çok hafif törpülüyoruz
final_test_preds = np.clip(final_test_preds, 0.05, 9.95)

submission = pd.DataFrame({'id': test['id'], 'bilissel_performans_skoru': final_test_preds})
submission.to_csv('submission_relative_v1.csv', index=False)
print("submission_relative_v1.csv başarıyla oluşturuldu!")


--- Eğitim Başlıyor (Seed: 42) ---
Seed 42 tamamlandı.

--- Eğitim Başlıyor (Seed: 2026) ---
Seed 2026 tamamlandı.

--- Eğitim Başlıyor (Seed: 1903) ---
Seed 1903 tamamlandı.

>>> YENİ OPTİMİZE CV RMSE: 1.21455 <<<
submission_relative_v1.csv başarıyla oluşturuldu!


In [1]:
import pandas as pd
import numpy as np
from catboost import CatBoostRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
import warnings

warnings.filterwarnings('ignore')

# --- 1. VERİ OKUMA ---
train = pd.read_csv('/kaggle/input/competitions/yzta-2026-datathon/train.csv')
test = pd.read_csv('/kaggle/input/competitions/yzta-2026-datathon/test_x.csv')

# --- 2. OPTİMİZE PREPROCESSING (SADE VE GÜÇLÜ) ---
def prepare_core_data(df, train_ref=None):
    df = df.copy()
    
    # Temizleme
    cat_cols = ['meslek', 'kronotip', 'ruh_sagligi_durumu']
    for col in cat_cols:
        df[col] = df[col].fillna('Bilinmiyor')
        
    if train_ref is not None:
        stres_med = train_ref.groupby('meslek')['stres_skoru'].median()
        df['stres_skoru'] = df['stres_skoru'].fillna(df['meslek'].map(stres_med))
        df['vucut_kitle_indeksi'] = df['vucut_kitle_indeksi'].fillna(train_ref['vucut_kitle_indeksi'].median())
    else:
        df['stres_skoru'] = df['stres_skoru'].fillna(df.groupby('meslek')['stres_skoru'].transform('median'))
        df['vucut_kitle_indeksi'] = df['vucut_kitle_indeksi'].fillna(df['vucut_kitle_indeksi'].median())
    
    df['uyku_oncesi_kafein_mg'] = df['uyku_oncesi_kafein_mg'].fillna(0)

    # ÖZELLİKLER (Sadece çalışanlar)
    df['total_quality_sleep'] = df['rem_yuzdesi'] + df['derin_uyku_yuzdesi']
    df['stres_krizi_siniri'] = (df['stres_skoru'] > 5.5).astype(int)
    df['is_stres_yukü'] = df['gunluk_calisma_saati'] * df['stres_skoru']
    
    # Göreceli Özellikler (En güçlü 2 tanesi)
    df['meslek_stres_farki'] = df['stres_skoru'] - df.groupby('meslek')['stres_skoru'].transform('mean')
    df['yas_uyku_verimi'] = df['total_quality_sleep'] / (df['yas'] + 1)
    
    return df

train = prepare_core_data(train)
test = prepare_core_data(test, train_ref=train)

# --- 3. EĞİTİM HAZIRLIĞI ---
cat_features = ['cinsiyet', 'meslek', 'ulke', 'kronotip', 'ruh_sagligi_durumu', 'mevsim', 'gun_tipi']
TARGET = 'bilissel_performans_skoru'

X = train.drop(['id', TARGET], axis=1)
y = train[TARGET]
X_test = test.drop(['id'], axis=1)

# Parametreleri "Güvenli Bölgeye" (Depth=7) geri çekiyoruz
cat_params = {
    'iterations': 5000,
    'learning_rate': 0.012,
    'depth': 7,
    'l2_leaf_reg': 5,
    'eval_metric': 'RMSE',
    'random_seed': 42,
    'verbose': 0,
    'early_stopping_rounds': 300,
    'bagging_temperature': 0.2
}

# --- 4. SEED AVERAGING ---
seeds = [42, 2026, 1903]
final_test_preds = np.zeros(len(X_test))
oof_preds = np.zeros(len(X))

for seed in seeds:
    print(f"Eğitiliyor (Seed: {seed})...")
    cat_params['random_seed'] = seed
    kf = KFold(n_splits=10, shuffle=True, random_state=seed)
    
    for fold, (train_idx, val_idx) in enumerate(kf.split(X, y)):
        X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]
        
        model = CatBoostRegressor(**cat_params)
        model.fit(X_tr, y_tr, eval_set=(X_val, y_val), cat_features=cat_features)
        
        oof_preds[val_idx] += model.predict(X_val) / len(seeds)
        final_test_preds += model.predict(X_test) / (len(seeds) * 10)

cv_rmse = np.sqrt(mean_squared_error(y, oof_preds))
print(f"\n>>> CORE CV RMSE: {cv_rmse:.5f} <<<")

# --- 5. KAYIT ---
# Hafif törpüleme RMSE dostudur
final_test_preds = np.clip(final_test_preds, 0.05, 9.95)
submission = pd.DataFrame({'id': test['id'], 'bilissel_performans_skoru': final_test_preds})
submission.to_csv('submission_core_final.csv', index=False)

Eğitiliyor (Seed: 42)...
Eğitiliyor (Seed: 2026)...
Eğitiliyor (Seed: 1903)...

>>> CORE CV RMSE: 1.21439 <<<
